# Практика 4 — REST API и JSON — проверенный справочник и запуск проекта

## Подготовка

In [ ]:
from pathlib import Path
import json
import hashlib
from copy import deepcopy
import requests
import pandas as pd
from pandas.testing import assert_frame_equal
DATA = Path("../data")
LOCAL = DATA
WORK = Path("../results")
WORK.mkdir(exist_ok=True)
decisions = pd.read_csv(
    DATA / "decisions.csv", dtype={"lei": "string", "county_code": "string"}
)
print("pandas", pd.__version__, decisions.shape)

## 01. Два формата одного источника

In [ ]:
import json, requests, hashlib
from copy import deepcopy
from pandas.testing import assert_frame_equal
manifest = json.loads((LOCAL / "manifest.json").read_text(encoding="utf-8"))
page1 = json.loads((LOCAL / "gleif_page_01.json").read_text(encoding="utf-8"))
page2 = json.loads((LOCAL / "gleif_page_02.json").read_text(encoding="utf-8"))
print(list(page1), len(page1["data"]), len(page2["data"]))

## 01. Два формата одного источника — шаг 2

In [ ]:
book = pd.read_excel(LOCAL / "hmda_exchange.xlsx", sheet_name=None, dtype={"lei":"string"})
print({name: frame.shape for name, frame in book.items()})

## A. Завершите чтение справочника

## 02. Полный пример нормализации

In [ ]:
first = page1["data"][0]
print(type(first), first["id"])
print(first["attributes"]["entity"]["legalName"]["name"])
flat_first = pd.json_normalize(page1["data"], sep=".")
flat_first[["id", "attributes.entity.legalName.name", "attributes.entity.status"]]

## 02. Полный пример нормализации — шаг 2

In [ ]:
aliases = pd.json_normalize(
    page1["data"], record_path=["attributes", "entity", "otherNames"], meta=["id"]
)
assert len(aliases) == sum(len(r["attributes"]["entity"]["otherNames"]) for r in page1["data"])
print(aliases.head(4))

## B. Самостоятельный справочник из двух страниц

## 03. GET без сети и настоящий сохранённый ответ

In [ ]:
endpoint = "https://api.gleif.org/api/v1/lei-records"
params = {"filter[lei]": ",".join(manifest["selected_lei"]), "page[size]":2}
prepared = requests.Request("GET", endpoint, params=params).prepare()
assert prepared.url == manifest["first_url"]
print(prepared.url)

## 03. GET без сети и настоящий сохранённый ответ — шаг 2

In [ ]:
response = requests.Response()
response.status_code = manifest["pages"][0]["status_code"]
response.headers["Content-Type"] = manifest["pages"][0]["content_type"]
response.url = manifest["pages"][0]["response_url"]
response._content = (LOCAL / "gleif_page_01.json").read_bytes()
response.encoding = "utf-8"

## 03. GET без сети и настоящий сохранённый ответ — шаг 3

In [ ]:
def decode_json_response(response):
    response.raise_for_status()
    media = response.headers.get("Content-Type", "").split(";")[0].lower()
    if media not in {"application/json", "application/vnd.api+json"}:
        raise ValueError("Ожидался JSON, получен " + media)
    payload = response.json()
    if not isinstance(payload, dict):
        raise ValueError("Корень должен быть объектом")
    if not isinstance(payload.get("data"), list):
        raise ValueError("Нет списка data")
    return payload

## 03. GET без сети и настоящий сохранённый ответ — шаг 4

In [ ]:
assert decode_json_response(response) == page1
print("Настоящее сохранённое тело прошло контракт")

## C. Обнаружьте два разных сбоя

## 04. Обход страниц с проверками

In [ ]:
cache = {info["request_url"]: json.loads((LOCAL / info["file"]).read_text(encoding="utf-8")) for info in manifest["pages"]}
def cached_loader(url):
    return cache[url]

## 04. Обход страниц с проверками — шаг 2

In [ ]:
def collect_pages(first_url, loader, max_pages=10):
    url = first_url
    seen_urls = set()
    records = []
    expected_total = None
    for page_number in range(max_pages):
        if url is None:
            break
        if url in seen_urls:
            raise ValueError("Цикл next")
        seen_urls.add(url)
        page = loader(url)
        if not isinstance(page.get("data"), list):
            raise ValueError("data должен быть списком")
        total = page["meta"]["pagination"]["total"]
        if expected_total is None:
            expected_total = total
        if total != expected_total:
            raise ValueError("Число записей изменилось между страницами")
        records.extend(page["data"])
        url = page["links"].get("next")
    if url is not None:
        raise ValueError("Предел страниц: получение не завершено")
    ids = [record["id"] for record in records]
    if len(ids) != len(set(ids)):
        raise ValueError("Повтор ключа между страницами")
    if len(records) != expected_total:
        raise ValueError("Неполный набор")
    return records

## 04. Обход страниц с проверками — шаг 3

In [ ]:
collected = collect_pages(manifest["first_url"], cached_loader)
assert len(collected) == 3
assert {r["id"] for r in collected} == set(manifest["selected_lei"])
assert len({p["golden_copy_publish_date"] for p in manifest["pages"]}) == 1
print("2 + 1 = 3; выбранные ключи получены")

## D. Самостоятельный аудит и сохранение

## 05. Покрытие JOIN и передача

In [ ]:
verified_lookup = pd.read_csv(LOCAL / "gleif_lookup.csv", dtype={"lei":"string"})
assert verified_lookup["lei"].is_unique
enriched = decisions.merge(verified_lookup, on="lei", how="left", validate="many_to_one", indicator=True)
assert len(enriched) == 6830
assert enriched["_merge"].eq("both").sum() == 76
assert enriched["_merge"].eq("left_only").sum() == 6754
assert_frame_equal(enriched[decisions.columns], decisions)
print(enriched["_merge"].value_counts())

## 05. Покрытие JOIN и передача — шаг 2

In [ ]:
enriched.to_csv(WORK / "enriched.csv", index=False)
summary = "# Получение GLEIF\n\nOffline; три выбранных LEI HMDA. Две реальные страницы, 3 организации.\n6830 записей сохранены; 76 совпадений. Текущий статус не описывает 2023.\nИсточник, время и хеши указаны в manifest.json из папки data/. Анализ воспроизводится из сохранённых bytes.\n"
(WORK / "README.md").write_text(summary, encoding="utf-8")

## E. Предложение командного проекта